# Experiment 4 — Variational Quantum Eigensolver (VQE)
Finds the ground state energy of a quantum Hamiltonian using a hybrid quantum-classical loop.

In [ ]:
!pip install qiskit qiskit-aer qiskit-algorithms matplotlib numpy scipy -q

## Imports

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Estimator
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt

## Step 1 — Define the Hamiltonian
A 2-qubit Hamiltonian that approximates the H2 molecule interaction.

In [ ]:
hamiltonian = SparsePauliOp.from_list([
    ("ZZ", -0.5),
    ("ZI", -0.5),
    ("IZ", -0.5),
    ("XX",  0.25),
    ("YY",  0.25),
])

exact_energies = np.linalg.eigvalsh(hamiltonian.to_matrix())
print("All energy levels:", np.round(exact_energies, 4))
print(f"Ground state energy (exact): {exact_energies[0]:.6f}")

## Step 2 — Build the Ansatz Circuit
A parameterized circuit whose parameters are tuned by the classical optimizer.

In [ ]:
def build_ansatz(num_qubits=2, layers=2):
    angles  = ParameterVector('theta', num_qubits * layers)
    circuit = QuantumCircuit(num_qubits)
    idx     = 0

    for layer in range(layers):
        for qubit in range(num_qubits):
            circuit.ry(angles[idx], qubit)
            idx += 1
        if layer < layers - 1:
            for qubit in range(num_qubits - 1):
                circuit.cx(qubit, qubit + 1)

    return circuit

ansatz = build_ansatz()
print("Ansatz Circuit:")
print(ansatz.draw('text'))
print(f"Number of parameters: {ansatz.num_parameters}")

## Step 3 — Cost Function
Runs the circuit on the quantum simulator and measures the energy expectation value.

In [ ]:
estimator  = Estimator()
energy_log = []

def cost(params):
    job    = estimator.run([ansatz], [hamiltonian], [params])
    energy = job.result().values[0]
    energy_log.append(float(energy))
    return float(energy)

## Step 4 — Run VQE Optimization

In [ ]:
np.random.seed(42)
starting_params = np.random.uniform(0, 2 * np.pi, ansatz.num_parameters)

print("Running VQE...")
vqe_result = minimize(cost, starting_params, method='COBYLA',
                      options={'maxiter': 500, 'rhobeg': 0.5})

print(f"\nVQE ground energy  : {vqe_result.fun:.6f}")
print(f"Exact ground energy: {exact_energies[0]:.6f}")
print(f"Error              : {abs(vqe_result.fun - exact_energies[0]):.6f}")

## Step 5 — Plot Convergence

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(energy_log, color='black', linewidth=1)
plt.axhline(y=exact_energies[0], color='gray', linestyle='--',
            label=f'Exact = {exact_energies[0]:.4f}')
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("VQE Convergence")
plt.legend()
plt.tight_layout()
plt.show()

## Step 6 — Inspect the Ground State

In [ ]:
final_circuit = ansatz.assign_parameters(vqe_result.x)
ground_state  = Statevector(final_circuit)

print("Ground state amplitudes:")
for i, amp in enumerate(ground_state.data):
    if abs(amp) > 0.01:
        label = format(i, f'0{ansatz.num_qubits}b')
        print(f"  |{label}>  amplitude={amp:.4f}  prob={abs(amp)**2:.4f}")